In [1]:
import pandas as pd

# Load the processed electricity price dataset
test_df = pd.read_csv('test_with_gas.csv')

# Load the CO2 emissions dataset
co2_df = pd.read_csv('co2_2024.csv')  # Replace with the actual file path
co2_df = co2_df.rename(columns={"Day": "date", "CO2 Emission Allowances, Auction DE": "co2_emissions"})

# Convert the 'date' column in CO2 data to datetime format
co2_df['date'] = pd.to_datetime(co2_df['date'], format='%d.%m.%Y')

# Fill missing CO2 emissions
co2_df['co2_emissions'] = co2_df['co2_emissions'].interpolate(method='linear')  # Linear interpolation for gaps
co2_df['co2_emissions'] = co2_df['co2_emissions'].ffill().bfill()  # Forward and backward fill


# Create a temporary date column in test DataFrame for merging
test_df['date'] = pd.to_datetime(test_df['ds']).dt.date  # Extract only the date from the 'ds' column
test_df['date'] = pd.to_datetime(test_df['date'])  # Ensure it is a datetime64[ns] type



In [2]:
co2_df.head()

,date,co2_emissions
0,2024-01-01,66.6
1,2024-01-02,66.6
2,2024-01-03,66.6
3,2024-01-04,66.6
4,2024-01-05,66.6


In [3]:
test_df.head()

,ds,y,unique_id,day_of_week,month,hour,is_weekend,is_holiday,month_sin,month_cos,day_of_week_sin,day_of_week_cos,hour_sin,hour_cos,gas_price,date
0,2024-01-01 00:00:00,0.10,electricity_prices,0,1,0,0,1,0.5,0.866025,0.0,1.0,0.000000,1.000000,31.574,2024-01-01
1,2024-01-01 01:00:00,0.01,electricity_prices,0,1,1,0,1,0.5,0.866025,0.0,1.0,0.258819,0.965926,31.574,2024-01-01
2,2024-01-01 02:00:00,0.00,electricity_prices,0,1,2,0,1,0.5,0.866025,0.0,1.0,0.500000,0.866025,31.574,2024-01-01
3,2024-01-01 03:00:00,-0.01,electricity_prices,0,1,3,0,1,0.5,0.866025,0.0,1.0,0.707107,0.707107,31.574,2024-01-01
4,2024-01-01 04:00:00,-0.03,electricity_prices,0,1,4,0,1,0.5,0.866025,0.0,1.0,0.866025,0.500000,31.574,2024-01-01


In [4]:
# Merge CO2 emissions into the electricity dataset using the temporary 'date' column
test_df = pd.merge(test_df, co2_df, how='left', left_on='date', right_on='date')

# Drop the temporary 'date' column (it was only used for merging)
test_df = test_df.drop(columns=['date'])

# Save the updated dataset to a new CSV file
test_df.to_csv('test_with_co2.csv', index=False)

print("CO2 emissions successfully added to the test dataset, with missing values handled by interpolation and forward/backward filling.")

CO2 emissions successfully added to the test dataset, with missing values handled by interpolation and forward/backward filling.
